In [8]:
import pandas as pd
import requests
import os
from datetime import datetime
import numpy as np
import pandas_ta as ta


In [22]:
symbol = 'ETHUSDT'
interval = '1h'
start_date = '2026-01-01'
end_date = '2026-04-30'
bb_lenth = 20
bb_std = 2.0

rsi_len = 14
rsi_upper_level = 70
rsi_lower_level = 30

adx_len = 14
adx_threshold = 25



In [4]:
def date_to_ms(date_str):
    """Конвертирует строку даты в миллисекунды (timestamp)"""
    dt = datetime.strptime(date_str, '%Y-%m-%d')
    return int(dt.timestamp() * 1000)

def download_candles_batch(symbol, interval, start_time=None, end_time=None, limit=1000):
    """Один запрос к Binance API за порцией свечей"""
    bin_url = "https://fapi.binance.com/fapi/v1/klines"
    params = {
        'symbol': symbol,
        'interval': interval,
        'limit': limit,
    }
    if start_time:
        params['startTime'] = start_time
    if end_time:
        params['endTime'] = end_time

    try:
        response = requests.get(bin_url, params=params, timeout=10)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"✗ Ошибка {response.status_code}: {response.text[:200]}")
            return None
    except requests.RequestException as e:
        print(f"✗ Сетевая ошибка: {e}")
        return None

def download_candles_range(symbol, interval, start_date, end_date, batch_size=1000):
    """
    Загружает свечи за период [start_date, end_date] порциями по batch_size.
    Возвращает список всех свечей в хронологическом порядке.
    """
    all_candles = []

    # Конвертируем даты в миллисекунды
    current_start = date_to_ms(start_date)
    end_time = date_to_ms(end_date)

    # Добавляем 1 день к end_date, чтобы включить последний день полностью
    from datetime import timedelta
    end_time += 24 * 60 * 60 * 1000  # +1 день в мс

    print(f"📥 Загрузка {symbol} {interval}: {start_date} → {end_date}")
    print(f"   Диапазон: {current_start} → {end_time} (мс)")

    batch_num = 0
    while current_start < end_time:
        batch_num += 1

        # Запрашиваем порцию
        candles = download_candles_batch(
            symbol, interval,
            start_time=current_start,
            end_time=end_time,
            limit=batch_size
        )

        if not candles or len(candles) == 0:
            print(f"   ⚠ Пустой ответ (порция #{batch_num}), завершаем")
            break

        # Добавляем свечи в общий список
        all_candles.extend(candles)

        # Обновляем current_start: последняя свеча + 1 мс (чтобы не дублировать)
        last_candle_close_time = candles[-1][6]  # поле 'Close time'
        current_start = last_candle_close_time + 1

        print(f"   ✓ Порция #{batch_num}: {len(candles)} свечей, последняя: {datetime.fromtimestamp(last_candle_close_time/1000)}")

        # Небольшая пауза, чтобы не получить бан от Binance (rate limit)
        if current_start < end_time:
            import time
            time.sleep(0.2)

    print(f"✅ Загружено всего: {len(all_candles)} свечей")
    return all_candles

def get_df(symbol, interval, start_date, end_date):
    """Получает свечи и возвращает отформатированный DataFrame"""

    candles = download_candles_range(symbol, interval, start_date, end_date)

    if not candles:
        print("✗ Не удалось получить данные")
        return None

    # Создаём DataFrame
    columns = [
        'Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume',
        'Close time', 'Quote asset volume', 'Number of trades',
        'Taker buy base asset volume', 'Taker buy quote asset volume',
        'Ignore'
    ]
    df = pd.DataFrame(candles, columns=columns)

    # Оставляем только нужные колонки
    cols_to_save = ['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume']
    df = df[cols_to_save].copy()

    # Преобразуем типы данных
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], unit='ms')
    numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
    df[numeric_cols] = df[numeric_cols].astype(float)

    # Сортируем по времени (на случай, если порции пришли не по порядку)
    df = df.sort_values('Timestamp').reset_index(drop=True)

    print(f"\n✓ DataFrame создан: {len(df)} строк")
    print(f"   Период: {df['Timestamp'].min()} → {df['Timestamp'].max()}")
    print(df.head(3))
    print("...")
    print(df.tail(3))

    return df

def save_df_to_csv(df: pd.DataFrame, symbol, interval, start_date=None, end_date=None, filepath='data'):
    """Сохраняет DataFrame в CSV"""
    if df is None or len(df) == 0:
        print("✗ DataFrame пустой!")
        return None

    os.makedirs(filepath, exist_ok=True)
    filename = f"{symbol}_{interval}_{start_date}_{end_date}.csv" if start_date and end_date else f"{symbol}_{interval}.csv"
    full_path = os.path.join(filepath, filename)

    try:
        df.to_csv(
            full_path,
            index=False,
            encoding='utf-8-sig',
            date_format='%Y-%m-%d %H:%M:%S',
            float_format='%.8f'
        )

        file_size = os.path.getsize(full_path) / 1024
        print(f"✓ Файл сохранён: {full_path}")
        print(f"  Строк: {len(df):,} | Размер: {file_size:.1f} KB")
        return full_path

    except Exception as e:
        print(f"✗ Ошибка сохранения: {e}")
        return None


NameError: name 'pd' is not defined

In [24]:
# ВСПОМОГАТЕЛЬНОЕ
def _enough(series, n=2):
    return series is not None and hasattr(series, "iloc") and series.dropna().shape[0] >= n

def _rolling_slope(series, window=10):
    """
    Грубая оценка наклона: (последнее - первое)/window.
    Используем как индикатор направления (положительный/отрицательный).
    """
    if len(series) < window or window < 2:
        return np.nan
    window_vals = series.iloc[-window:]
    return (window_vals.iloc[-1] - window_vals.iloc[0]) / (window - 1 + 1e-9)


In [27]:
'''
Стратегия:  Возврат к средней по полосам боллинджера с фильтрами ADX и RSI

Суть: Ловим отскок цены к среднему после резких импульсов. Покупаем, когда цена «перерастянута» вниз, и продаем, когда она улетела слишком высоко вверх. Работаем в боковике.

Где торгуем:
- Binance (фьючерсы или спот)
- Топ-50 монет по капитализации
- Таймфрейм: 1 час
- Лонг и Шорт

Индикаторы:
RSI (9-14) — показывает, когда цена «перегрета» или «перепродана»
Bollinger Bands (20, 2.0) — показывает границы нормального движения цены
ADX (14) — фильтр: не входим, если на рынке сильный тренд
ATR (14) — для расчёта стопов

Вход в сделку (Лонг):
- Цена закрылась ниже нижней полосы Боллинджера
- RSI упал ниже 20 (зона перепроданности)
- ADX ниже 25 (рынок не в сильном тренде)
- Входим на открытии следующей свечи

Для Шорта — всё зеркально: цена выше верхней полосы, RSI > 80, ADX < 25.

Риск-менеджмент:
- Стоп-лосс: сразу после входа ставим на 1.5 × ATR от цены входа
- Перенос стопа в Безубыток: когда цена прошла в нашу сторону X × ATR (параметр для подбора), переносим стоп на точку входа + комиссии (~0.3%)
- трейлинг стоп (описан дальше)

Выход из позиции:
- 50% позиции закрываем, когда цена касается средней линии Боллинджера. В этот момент активируем трейлинг для остатка:
- Фиксируем расстояние между ценой закрытия первой части и текущим стопом
- Дальше стоп двигается за ценой, сохраняя это расстояние
- Если цена разворачивается — остаток закрывается по стопу
- Если цена идёт дальше — стоп подтягивается
- Остаток 50% закрываем, когда цена достигает противоположной полосы Боллинджера ИЛИ срабатывает трейлинг-стоп

Что хотим оптимизировать:
- Множитель для перевода в безубыток (break_even_atr_multiplier)
- Уровни RSI для входа
- Порог ADX для фильтрации тренда
- мультипликатор ATR для первоначального стопа
'''


def get_signals(df):
    c = df['Close']; h, l = df['High'], df['Low']; v = df.get('Volume')
    adx = ta.adx(h, l, c, length=adx_len).filter(like='ADX_').iloc[:, 0]
    if not _enough(adx, 1):
        return None
    bb = ta.bbands(c, length=bb_lenth, lower_std=bb_std, upper_std=bb_std)
    rsi = ta.rsi(c, length=rsi_len)
    if bb is None or not _enough(rsi, 1):
        return None
    bbl, bbu, bbm = bb.iloc[:, 0], bb.iloc[:, 2],bb.iloc[:, 1]

    # Сами сигналы
    buy = (c.iloc[-1] <= bbl.iloc[-1]) and (rsi.iloc[-1] < rsi_upper_level) and adx[-1] < adx_threshold
    sell = (c.iloc[-1] >= bbu.iloc[-1]) and (rsi.iloc[-1] > rsi_lower_level) and adx[-1] < adx_threshold
    if buy:
        print ("Покупка")
        return "Покупка"
    if sell:
        print ("Продажа")
        return "Продажа"
    print ('No signal')
    return None

In [28]:
# ==================== ЗАПУСК ====================

if __name__ == "__main__":
    df = get_df(symbol=symbol, interval=interval, start_date=start_date, end_date=end_date)
    # save_df_to_csv(df=df, symbol=symbol, start_date=start_date, end_date=end_date, interval=interval)
    get_signals(df)


📥 Загрузка ETHUSDT 1h: 2026-01-01 → 2026-04-30
   Диапазон: 1767214800000 → 1777582800000 (мс)
   ✓ Порция #1: 1000 свечей, последняя: 2026-02-11 15:59:59.999000
   ✓ Порция #2: 1000 свечей, последняя: 2026-03-25 07:59:59.999000
   ✓ Порция #3: 881 свечей, последняя: 2026-05-01 00:59:59.999000
✅ Загружено всего: 2881 свечей

✓ DataFrame создан: 2881 строк
   Период: 2025-12-31 21:00:00 → 2026-04-30 21:00:00
            Timestamp     Open     High      Low    Close     Volume
0 2025-12-31 21:00:00  2974.62  2984.05  2973.52  2981.04  48500.525
1 2025-12-31 22:00:00  2981.04  2983.37  2973.31  2976.07  26571.778
2 2025-12-31 23:00:00  2976.06  2976.06  2969.01  2970.40  27142.577
...
               Timestamp     Open     High      Low    Close     Volume
2878 2026-04-30 19:00:00  2258.58  2265.91  2256.80  2260.70  50829.400
2879 2026-04-30 20:00:00  2260.71  2266.35  2259.20  2262.83  33570.374
2880 2026-04-30 21:00:00  2262.83  2263.50  2252.45  2255.09  43573.036
No signal


In [16]:
def date_to_ms(date_str):
    """Конвертирует строку даты в миллисекунды (timestamp)"""
    dt = datetime.strptime(date_str, '%Y-%m-%d')
    return int(dt.timestamp() * 1000)

def download_candles_batch(symbol, interval, start_time=None, end_time=None, limit=1000):
    """Один запрос к Binance API за порцией свечей"""
    bin_url = "https://fapi.binance.com/fapi/v1/klines"
    params = {
        'symbol': symbol,
        'interval': interval,
        'limit': limit,
    }
    if start_time:
        params['startTime'] = start_time
    if end_time:
        params['endTime'] = end_time

    try:
        response = requests.get(bin_url, params=params, timeout=10)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"✗ Ошибка {response.status_code}: {response.text[:200]}")
            return None
    except requests.RequestException as e:
        print(f"✗ Сетевая ошибка: {e}")
        return None

def download_candles_range(symbol, interval, start_date, end_date, batch_size=1000):
    """
    Загружает свечи за период [start_date, end_date] порциями по batch_size.
    Возвращает список всех свечей в хронологическом порядке.
    """
    all_candles = []

    # Конвертируем даты в миллисекунды
    current_start = date_to_ms(start_date)
    end_time = date_to_ms(end_date)

    # Добавляем 1 день к end_date, чтобы включить последний день полностью
    from datetime import timedelta
    end_time += 24 * 60 * 60 * 1000  # +1 день в мс

    print(f"📥 Загрузка {symbol} {interval}: {start_date} → {end_date}")
    print(f"   Диапазон: {current_start} → {end_time} (мс)")

    batch_num = 0
    while current_start < end_time:
        batch_num += 1

        # Запрашиваем порцию
        candles = download_candles_batch(
            symbol, interval,
            start_time=current_start,
            end_time=end_time,
            limit=batch_size
        )

        if not candles or len(candles) == 0:
            print(f"   ⚠ Пустой ответ (порция #{batch_num}), завершаем")
            break

        # Добавляем свечи в общий список
        all_candles.extend(candles)

        # Обновляем current_start: последняя свеча + 1 мс (чтобы не дублировать)
        last_candle_close_time = candles[-1][6]  # поле 'Close time'
        current_start = last_candle_close_time + 1

        print(f"   ✓ Порция #{batch_num}: {len(candles)} свечей, последняя: {datetime.fromtimestamp(last_candle_close_time/1000)}")

        # Небольшая пауза, чтобы не получить бан от Binance (rate limit)
        if current_start < end_time:
            import time
            time.sleep(0.2)


    print(f"✅ Загружено всего: {len(all_candles)} свечей")
    return all_candles[:-1]

In [17]:
download_candles_range(symbol='BTCUSDT',interval='1h', start_date='2026-05-18', end_date='2026-05-19', batch_size=1000)

📥 Загрузка BTCUSDT 1h: 2026-05-18 → 2026-05-19
   Диапазон: 1779051600000 → 1779224400000 (мс)
   ✓ Порция #1: 35 свечей, последняя: 2026-05-19 10:59:59.999000
   ⚠ Пустой ответ (порция #2), завершаем
✅ Загружено всего: 35 свечей


[[1779051600000,
  '78230.10',
  '78453.00',
  '78190.60',
  '78365.40',
  '2194.705',
  1779055199999,
  '171979051.18210',
  46907,
  '1252.254',
  '98132704.56750',
  '0'],
 [1779055200000,
  '78365.40',
  '78372.80',
  '77777.00',
  '77961.60',
  '5643.159',
  1779058799999,
  '440247034.43870',
  155918,
  '2715.288',
  '211822372.90820',
  '0'],
 [1779058800000,
  '77961.70',
  '77970.00',
  '76666.00',
  '77430.40',
  '24982.540',
  1779062399999,
  '1930193869.32180',
  410512,
  '9801.427',
  '757361154.39950',
  '0'],
 [1779062400000,
  '77430.40',
  '77452.70',
  '76875.00',
  '77175.70',
  '10777.547',
  1779065999999,
  '831081634.29670',
  305724,
  '5494.842',
  '423696217.37630',
  '0'],
 [1779066000000,
  '77175.70',
  '77274.80',
  '76700.00',
  '77084.50',
  '10313.877',
  1779069599999,
  '793500745.40060',
  259991,
  '5202.232',
  '400288687.94680',
  '0'],
 [1779069600000,
  '77084.50',
  '77190.00',
  '76668.80',
  '76809.40',
  '5970.755',
  1779073199999,
  '4